# Deep-Space Link Agent — Track 1
**Matlab in Space Hackathon · Deep-Space Communication & Signal Intelligence**

An autonomous receiver that finds a weak, drifting BPSK telemetry signal buried in noise, tracks Doppler,
survives dropouts and frequency hops, and outputs decoded, CRC-checked spacecraft telemetry — with **no manual tuning**.

```
synthetic telemetry ─► BPSK transmitter ─► deep-space channel ─► autonomous receiver agent ─► telemetry log
```

Run the cells from top to bottom (**Kernel → Restart & Run All**).

In [ ]:
# If needed: !pip install numpy scipy matplotlib pandas
import struct
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
SPS = 8          # samples per symbol

## 1. Telemetry frame format
CCSDS-style framing, simplified:

`[ ASM sync marker 0x1ACFFC1D (32 bits) | randomized( payload 13 B + CRC-16 2 B ) ]`

- **Payload:** sequence number, time, temperature, bus voltage, current, status flags
- **Randomizer (LFSR):** breaks up long runs of identical bits so the timing and carrier loops stay locked
- **CRC-16:** tells us whether a decoded frame can be trusted

In [ ]:
ASM = 0x1ACFFC1D                     # CCSDS attached sync marker
PAYLOAD_FMT = ">HIhHhB"              # seq, time_ms, temp_cC, bus_mV, current_mA, status
PAYLOAD_LEN = struct.calcsize(PAYLOAD_FMT)          # 13 bytes
BODY_BITS = (PAYLOAD_LEN + 2) * 8                    # payload + CRC
FRAME_BITS = 32 + BODY_BITS
FIELDS = ["seq", "time_ms", "temp_C", "bus_V", "current_A", "status"]


def int_to_bits(value, n):
    return np.array([(value >> (n - 1 - i)) & 1 for i in range(n)], dtype=np.uint8)


ASM_BITS = int_to_bits(ASM, 32)


def crc16_ccitt(data: bytes) -> int:
    crc = 0xFFFF
    for b in data:
        crc ^= b << 8
        for _ in range(8):
            crc = ((crc << 1) ^ 0x1021) if crc & 0x8000 else (crc << 1)
            crc &= 0xFFFF
    return crc


def _pn_sequence(nbits):
    """LFSR randomizer (x^8+x^7+x^5+x^3+1, seed all ones). Breaks up long runs
    of identical bits so timing/carrier loops stay locked."""
    state = [1] * 8
    out = np.empty(nbits, dtype=np.uint8)
    for i in range(nbits):
        out[i] = state[0]
        fb = state[0] ^ state[3] ^ state[5] ^ state[7]
        state = state[1:] + [fb]
    return out


PN = _pn_sequence(BODY_BITS)


def make_packet(seq, rng):
    """Synthetic spacecraft housekeeping telemetry."""
    t_ms = seq * 1000
    temp = 21.0 + 4.0 * np.sin(seq / 25) + rng.normal(0, 0.1)
    bus_v = 28.0 - 0.002 * seq + rng.normal(0, 0.02)
    current = 1.8 + 0.3 * np.sin(seq / 10) + rng.normal(0, 0.02)
    status = 0b0000_0001 | ((seq % 50 == 0) << 3)        # bit3 = heartbeat flag
    return struct.pack(PAYLOAD_FMT, seq & 0xFFFF, t_ms, int(temp * 100),
                       int(bus_v * 1000), int(current * 1000), status)


def build_frame(payload: bytes) -> np.ndarray:
    body = payload + struct.pack(">H", crc16_ccitt(payload))
    body_bits = np.unpackbits(np.frombuffer(body, dtype=np.uint8)) ^ PN
    return np.concatenate([ASM_BITS, body_bits])


def decode_body(body_bits: np.ndarray):
    """Returns (dict, crc_ok)."""
    raw = np.packbits((body_bits.astype(np.uint8) ^ PN)).tobytes()
    payload, crc = raw[:PAYLOAD_LEN], struct.unpack(">H", raw[PAYLOAD_LEN:])[0]
    ok = crc16_ccitt(payload) == crc
    seq, t_ms, temp, bus, cur, status = struct.unpack(PAYLOAD_FMT, payload)
    return dict(seq=seq, time_ms=t_ms, temp_C=temp / 100, bus_V=bus / 1000,
                current_A=cur / 1000, status=status), ok

In [ ]:
# quick self-test: build a frame and decode it again
rng = np.random.default_rng(0)
frame = build_frame(make_packet(7, rng))
print("frame length:", len(frame), "bits")
print(decode_body(frame[32:]))

## 2. Transmitter: BPSK with root-raised-cosine (RRC) pulse shaping
Bit 0 → +1, bit 1 → −1. The RRC filter limits the bandwidth; the receiver uses the same filter as a matched filter.

In [ ]:
def rrc_taps(sps, beta=0.35, span=10):
    t = np.arange(-span * sps / 2, span * sps / 2 + 1) / sps
    h = np.zeros_like(t)
    for i, ti in enumerate(t):
        if np.isclose(ti, 0):
            h[i] = 1 - beta + 4 * beta / np.pi
        elif np.isclose(abs(ti), 1 / (4 * beta)):
            h[i] = beta / np.sqrt(2) * ((1 + 2 / np.pi) * np.sin(np.pi / (4 * beta))
                                        + (1 - 2 / np.pi) * np.cos(np.pi / (4 * beta)))
        else:
            h[i] = (np.sin(np.pi * ti * (1 - beta)) + 4 * beta * ti * np.cos(np.pi * ti * (1 + beta))) \
                   / (np.pi * ti * (1 - (4 * beta * ti) ** 2))
    return h / np.sqrt(np.sum(h ** 2))


def transmit(n_frames, sps=8, beta=0.35, seed=0):
    """Returns (baseband IQ, list of payloads, bit stream)."""
    rng = np.random.default_rng(seed)
    payloads = [make_packet(k, rng) for k in range(n_frames)]
    bits = np.concatenate([build_frame(p) for p in payloads])
    symbols = 1.0 - 2.0 * bits                      # bit 0 -> +1, bit 1 -> -1
    up = np.zeros(len(symbols) * sps)
    up[::sps] = symbols
    x = np.convolve(up, rrc_taps(sps, beta), mode="same") * np.sqrt(sps)
    return x.astype(np.complex128), payloads, bits

## 3. Deep-space channel
Impairments the agent must handle without being told their values:

| Impairment | Why it happens in space |
|---|---|
| Doppler offset + linear drift | spacecraft and Earth moving relative to each other |
| Frequency hop | sudden oscillator or transponder change |
| Dropout | occultation, antenna pointing loss, fades |
| Phase noise, clock drift | imperfect oscillators on both ends |
| AWGN | weak signal over huge distances |
| Silence before/after | the receiver has to *find* the pass itself |

In [ ]:
def deep_space_channel(x, sps, esn0_db=8.0, f0=0.05, doppler_rate=2e-8,
                       hop_at=None, hop_df=0.0, dropouts=(), clock_ppm=50.0,
                       phase_noise_std=2e-3, lead_in=20000, tail=20000, seed=1):
    rng = np.random.default_rng(seed)
    # silence before/after the pass (receiver must find the signal itself)
    x = np.concatenate([np.zeros(lead_in), x, np.zeros(tail)]).astype(complex)
    n = len(x)

    # sample-clock drift (TX/RX oscillator mismatch)
    t = np.arange(n) * (1 + clock_ppm * 1e-6) + 0.37
    t = t[t < n - 1]
    x = np.interp(t, np.arange(n), x.real) + 1j * np.interp(t, np.arange(n), x.imag)
    n = len(x)

    # carrier: Doppler offset, linear Doppler drift, optional hop, phase noise
    k = np.arange(n)
    f = f0 + doppler_rate * k
    if hop_at is not None:
        f[hop_at:] += hop_df
    phase = 2 * np.pi * np.cumsum(f) + np.cumsum(rng.normal(0, phase_noise_std, n))
    x = x * np.exp(1j * (phase + rng.uniform(0, 2 * np.pi)))

    for start, length in dropouts:                 # signal fades / occultations
        x[start:start + length] = 0

    # AWGN at the requested Es/N0
    active = np.abs(x) > 0
    es = np.mean(np.abs(x[active]) ** 2) * sps
    n0 = es / 10 ** (esn0_db / 10)
    noise = np.sqrt(n0 / 2) * (rng.normal(size=n) + 1j * rng.normal(size=n))
    truth = dict(f=f, lead_in=lead_in, dropouts=list(dropouts), hop_at=hop_at)
    return x + noise, truth

## 4. The autonomous receiver agent
1. **Activity detection:** blind SNR per block (the median FFT bin estimates the noise floor) → signal segments
2. **Doppler:** squaring BPSK produces a pure tone at 2·f_c → FFT peak per block → hop detection → robust drift fit → de-rotation
3. **Matched filter + AGC**
4. **Gardner timing recovery**
5. **Costas carrier loop + lock detector**
6. **Frame sync** on the ASM (also resolves the 180° phase ambiguity) → de-randomize → CRC → telemetry

In [ ]:
import time


def loop_gains(bn, zeta=0.707, kd=1.0):
    theta = bn / (zeta + 1 / (4 * zeta))
    d = 1 + 2 * zeta * theta + theta ** 2
    return 4 * zeta * theta / (d * kd), 4 * theta ** 2 / (d * kd)


class SignalAgent:
    def __init__(self, sps=8, beta=0.35, block=1024, snr_gate_db=-12.0,
                 fft_block=4096, hop_thresh=2e-3, classifier=None, verbose=True):
        self.sps, self.beta = sps, beta
        self.block, self.snr_gate = block, 10 ** (snr_gate_db / 10)
        self.fft_block, self.hop_thresh = fft_block, hop_thresh
        self.events, self.verbose = [], verbose
        self.classifier = classifier          # optional ModulationClassifier trained on RadioML
        self.rrc = rrc_taps(sps, beta)

    def log(self, sample, msg):
        self.events.append((int(sample), msg))
        if self.verbose:
            print(f"[agent @ {int(sample):>8d}] {msg}")

    def say(self, msg):
        """Progress message only (printed when verbose, not stored as an event)."""
        if self.verbose:
            print(f"      ... {msg}")

    # ---------- 1. where is there a signal? ----------
    def blind_snr(self, x):
        """Per-block SNR without knowing the carrier: the signal occupies
        < 50% of the band, so the median FFT bin is a noise-floor estimate."""
        nb = len(x) // self.block
        blocks = x[:nb * self.block].reshape(nb, self.block) * np.hanning(self.block)
        p = np.abs(np.fft.fft(blocks, axis=1)) ** 2
        noise_bin = np.median(p, axis=1) / np.log(2)      # median of exp. dist = ln2*mean
        snr = (p.sum(1) - noise_bin * self.block) / (noise_bin * self.block)
        return np.maximum(snr, 1e-6)

    def find_segments(self, x, min_blocks=8):
        snr = self.blind_snr(x)
        active = snr > self.snr_gate
        segs, start = [], None
        for i, a in enumerate(np.append(active, False)):
            if a and start is None:
                start = i
            elif not a and start is not None:
                if i - start >= min_blocks:
                    segs.append((start * self.block, i * self.block))
                start = None
        return segs, snr

    # ---------- 2. Doppler: offset, drift, hops ----------
    def doppler_track(self, x):
        """BPSK squared has a pure tone at 2*f_c -> FFT peak gives f_c."""
        n, step, nfft = self.fft_block, self.fft_block // 2, 4 * self.fft_block
        centers, f = [], []
        for s in range(0, len(x) - n + 1, step):
            sq = (x[s:s + n] * np.hanning(n)) ** 2
            spec = np.abs(np.fft.fft(sq, nfft))
            k = np.argmax(spec)
            # parabolic interpolation for sub-bin accuracy
            a, b, c = spec[k - 1], spec[k], spec[(k + 1) % nfft]
            k_frac = k + 0.5 * (a - c) / (a - 2 * b + c)
            f2 = np.fft.fftfreq(nfft)[k] + (k_frac - k) / nfft
            centers.append(s + n / 2)
            f.append(f2 / 2)
        return np.array(centers), np.array(f)

    def split_on_hops(self, centers, f, seg_start):
        cuts = [0]
        for i in range(1, len(f) - 1):
            # a real hop: big jump that persists (not a single noisy outlier)
            if i - cuts[-1] < 2:
                continue
            if abs(f[i] - f[i - 1]) > self.hop_thresh and abs(f[i + 1] - f[i - 1]) > self.hop_thresh:
                cuts.append(i)
                self.log(seg_start + centers[i], f"frequency hop detected: "
                         f"{f[i-1]:+.4f} -> {f[i+1]:+.4f} cyc/sample, re-acquiring")
        cuts.append(len(f))
        return [(cuts[j], cuts[j + 1]) for j in range(len(cuts) - 1)]

    def fit_doppler(self, c, f):
        """Robust linear fit f(t) = f0 + rate*t (outlier rejection by MAD)."""
        if len(f) < 3:
            return np.median(f), 0.0
        keep = np.abs(f - np.median(f)) < 5 * (np.median(np.abs(f - np.median(f))) + 1e-6)
        rate, f0 = np.polyfit(c[keep], f[keep], 1)
        return f0, rate

    # ---------- 3-5. demodulation ----------
    def gardner(self, y, bn=0.005):
        kp, ki = loop_gains(bn, kd=2.0)
        sps, out = self.sps, []
        t, integ = float(sps), 0.0
        yl = y.tolist()                          # python list: fast scalar access in the loop

        def interp(tt):                          # linear interpolation between samples
            i = int(tt)
            fr = tt - i
            return yl[i] + (yl[i + 1] - yl[i]) * fr
        prev = interp(t - sps)
        while t + sps < len(y) - 1:
            cur, mid = interp(t), interp(t - sps / 2)
            e = np.real((prev - cur) * np.conj(mid))
            integ += ki * e
            t += sps + np.clip(kp * e + integ, -sps / 4, sps / 4)
            out.append(cur)
            prev = cur
        return np.array(out)

    def costas(self, s, bn=0.02):
        kp, ki = loop_gains(bn)
        phase, integ = 0.0, 0.0
        out = np.empty_like(s)
        for i, v in enumerate(s):
            z = v * np.exp(-1j * phase)
            out[i] = z
            e = np.sign(z.real) * z.imag
            integ += ki * e
            phase += kp * e + integ
        return out

    @staticmethod
    def lock_metric(sym, win=64):
        r2, i2 = sym.real ** 2, sym.imag ** 2
        k = np.ones(win) / win
        return np.convolve(r2 - i2, k, "same") / (np.convolve(r2 + i2, k, "same") + 1e-12)

    # ---------- 6. frames ----------
    def frame_sync(self, sym, thresh=0.6):
        soft = sym.real / (np.mean(np.abs(sym.real)) + 1e-12)
        asm = 1.0 - 2.0 * ASM_BITS
        corr = np.correlate(soft, asm, "valid") / len(asm)
        frames, i = [], 0
        while i < len(corr) - FRAME_BITS + 32:
            if abs(corr[i]) > thresh:
                j = i + np.argmax(np.abs(corr[i:i + 8]))          # local peak
                pol = np.sign(corr[j])                            # 180° ambiguity
                body = soft[j + 32:j + FRAME_BITS] * pol
                if len(body) == FRAME_BITS - 32:
                    frames.append((j, (body < 0).astype(np.uint8), abs(corr[j])))
                i = j + FRAME_BITS - 8
            else:
                i += 1
        return frames

    # ---------- the agent ----------
    def run(self, x):
        self.say(f"scanning {len(x)} samples for signal activity (blind SNR per {self.block}-sample block)")
        segs, snr = self.find_segments(x)
        self.say(f"found {len(segs)} active segment(s)")
        self.diag = dict(block_snr=snr, doppler=[], symbols=[], lock=[])
        if not segs:
            self.log(0, "no signal found")
        records = []
        for seg_id, (s0, s1) in enumerate(segs):
            seg_snr = 10 * np.log10(np.median(snr[s0 // self.block:s1 // self.block]))
            self.log(s0, f"signal acquired (segment {seg_id}, blind SNR ≈ {seg_snr:.1f} dB full-band)")
            xs = x[s0:s1]
            self.say(f"estimating Doppler on {len(xs)} samples (squaring + FFT per {self.fft_block}-sample block)")
            c, f = self.doppler_track(xs)
            for a, b in self.split_on_hops(c, f, s0):
                lo = 0 if a == 0 else int(c[a] - self.fft_block / 2)
                hi = len(xs) if b == len(f) else int(c[b - 1] + self.fft_block / 2)
                f0, rate = self.fit_doppler(c[a:b], f[a:b])
                self.log(s0 + lo, f"Doppler {f0 + rate*lo:+.5f} cyc/sample, drift {rate:+.2e} /sample")
                k = np.arange(lo, hi)
                phase = 2 * np.pi * (f0 * k + 0.5 * rate * k ** 2)
                z = xs[lo:hi] * np.exp(-1j * phase)
                self.diag["doppler"].append((s0 + c[a:b], f[a:b], s0 + k, f0 + rate * k))

                if self.classifier is not None:
                    label, conf, top = self.classifier.classify_signal(z)
                    self.diag.setdefault("modulation", []).append((s0 + lo, label, conf, top))
                    self.log(s0 + lo, f"modulation identified: {label} ({100*conf:.0f}% of windows)")
                    if label != "BPSK":
                        self.log(s0 + lo, f"no {label} demodulator yet, trying BPSK anyway")

                self.say(f"demodulating {hi - lo} samples (~{(hi - lo) // self.sps} symbols)")
                t0 = time.time()
                y = np.convolve(z, self.rrc, "same")
                y /= np.sqrt(np.mean(np.abs(y) ** 2))
                self.diag.setdefault("filtered", []).append(y)
                self.say("matched filter + AGC done, running Gardner timing recovery...")
                sym = self.gardner(y)
                sym /= np.mean(np.abs(sym))
                self.say(f"timing recovered {len(sym)} symbols ({time.time()-t0:.1f} s), running Costas carrier loop...")
                sym = self.costas(sym)
                lock = self.lock_metric(sym)
                self.diag["symbols"].append(sym)
                self.diag["lock"].append(lock)
                locked = np.mean(lock[len(lock) // 4:] > 0.3)
                self.log(s0 + hi, f"sub-segment done, carrier locked {100*locked:.0f}% of time")

                found = self.frame_sync(sym)
                self.say(f"frame sync found {len(found)} sync markers, checking CRC...")
                n_ok = 0
                for idx, body, q in found:
                    rec, ok = decode_body(body)
                    n_ok += ok
                    rec.update(crc_ok=ok, segment=seg_id, rx_sample=s0 + lo + idx * self.sps,
                               sync_quality=round(q, 3))
                    records.append(rec)
                self.say(f"{n_ok}/{len(found)} frames passed CRC ({time.time()-t0:.1f} s for this sub-segment)")
            self.log(s1, f"signal lost (end of segment {seg_id}) — returning to search")
        return records

## 5. Run one scenario
Doppler offset 0.05 cycles/sample with drift, a dropout at about 35% of the pass, and a +0.03 frequency hop at about 68%.

In [ ]:
ESN0_DB = 8.0     # try 5 or 6 to make it harder
N_FRAMES = 300

def scenario(esn0, n_frames, verbose=True):
    tx, payloads, _ = transmit(n_frames, sps=SPS)
    n = len(tx) + 20000
    rx, truth = deep_space_channel(
        tx, SPS, esn0_db=esn0, f0=0.05, doppler_rate=2e-8,
        hop_at=int(0.68 * n), hop_df=0.03,
        dropouts=[(int(0.35 * n), 25000)])
    agent = SignalAgent(sps=SPS, verbose=verbose)
    recs = agent.run(rx)
    good = {r["seq"] for r in recs if r["crc_ok"]}
    return rx, truth, agent, recs, len(good & set(range(n_frames))) / n_frames

rx, truth, agent, recs, success = scenario(ESN0_DB, N_FRAMES)

## 6. Telemetry log

In [ ]:
log = pd.DataFrame(recs)
log.to_csv("telemetry_log.csv", index=False)
print(f"frames found: {len(log)} | CRC ok: {log.crc_ok.sum()} | "
      f"unique frames recovered: {100*success:.1f}% of {N_FRAMES}")
log[log.crc_ok].head(10)

In [ ]:
ok = log[log.crc_ok]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
for a, col, unit in zip(ax, ["temp_C", "bus_V", "current_A"], ["°C", "V", "A"]):
    a.plot(ok.time_ms / 1000, ok[col], ".", ms=3)
    a.set(title=col, xlabel="mission time [s]", ylabel=unit); a.grid(alpha=.3)
plt.suptitle("Decoded telemetry (gaps = dropout / hop)"); plt.tight_layout(); plt.show()

## 7. Before / after plots

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

# spectrum before / after
f_b, p_b = welch(rx, nperseg=2048, return_onesided=False)
f_a, p_a = welch(agent.diag["filtered"][0], nperseg=2048, return_onesided=False)
for fr, pp, lab in [(f_b, p_b, "before: raw received IQ"),
                    (f_a, p_a, "after: agent's Doppler removal + matched filter")]:
    o = np.argsort(fr)
    ax[0, 0].plot(fr[o], 10*np.log10(pp[o] / pp.max()), label=lab, alpha=.8)
ax[0, 0].set(title="Spectrum before / after", xlabel="frequency [cycles/sample]", ylabel="PSD [dB, normalized]")
ax[0, 0].legend(); ax[0, 0].grid(alpha=.3)

# spectrogram + Doppler track
ax[0, 1].specgram(rx, NFFT=1024, noverlap=512, Fs=1, cmap="viridis")
for (cb, fb, k, fit) in agent.diag["doppler"]:
    ax[0, 1].plot(k[::200], fit[::200], "r-", lw=1.5)
ax[0, 1].set(title="Spectrogram + agent's Doppler track (red)", xlabel="sample", ylabel="cycles/sample")

# constellation
s0 = truth["lead_in"]
naive = rx[s0:s0 + 4000 * SPS:SPS]
sym = agent.diag["symbols"][0][500:4500]
ax[1, 0].plot(naive.real, naive.imag, ".", ms=2, alpha=.4, label="before (raw samples)")
ax[1, 0].plot(sym.real, sym.imag, ".", ms=2, alpha=.4, label="after (agent output)")
ax[1, 0].set(title="Constellation", aspect="equal", xlim=(-3, 3), ylim=(-3, 3)); ax[1, 0].legend()

# blind SNR + agent events
ax[1, 1].plot(np.arange(len(agent.diag["block_snr"])) * agent.block, 10*np.log10(agent.diag["block_snr"]), lw=.8)
ax[1, 1].axhline(10*np.log10(agent.snr_gate), color="r", ls="--", label="acquisition gate")
for smp, msg in agent.events:
    ax[1, 1].axvline(smp, color="k", alpha=.25)
ax[1, 1].set(title="Blind SNR + agent events (grey lines)", xlabel="sample", ylabel="dB"); ax[1, 1].legend()
plt.tight_layout(); plt.savefig("overview.png", dpi=130); plt.show()

In [ ]:
# the agent's decision log (useful for the README / slides)
pd.DataFrame(agent.events, columns=["sample", "event"])

## 8. Performance vs SNR
The curve levels off around 77–83% because the deliberate dropout removes about 17% of frames.
Takes about a minute to run.

In [ ]:
snrs = np.arange(0, 13, 1.0)
rates = [scenario(e, 120, verbose=False)[-1] for e in snrs]
plt.figure(figsize=(6, 4))
plt.plot(snrs, 100 * np.array(rates), "o-")
plt.xlabel("Es/N0 [dB]"); plt.ylabel("frames decoded (CRC ok) [%]")
plt.title("Frame success vs SNR (Doppler + drift + hop + dropout)")
plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("sweep.png", dpi=130); plt.show()

## 9. Next steps
- [ ] Forward error correction (convolutional code + Viterbi) to decode below ~4 dB
- [ ] RadioML 2016.10A modulation classifier (README Part 1), then add a QPSK demodulator
- [ ] Harsher scenarios: faster Doppler, several hops, short dropouts; measure re-acquisition time
- [ ] Real SatNOGS IQ recording (README Part 3)

**Credits:** RadioML — DeepSig Inc., CC BY-NC-SA 4.0 (O'Shea & West, GRCon 2016). SatNOGS — Libre Space Foundation.